# Time-dependent delivery routing for ADEME — Modeling deliverable

**CesiCDP project team** · Response to ADEME's call for new mobility solutions

CesiCDP wants to show ADEME that Operations Research can reduce the time, energy and emissions of delivery rounds. In this first deliverable we define precisely **which problem we solve** and **how hard it is**, before writing any algorithm.

**Roadmap**
1. Context and problem statement
2. Data and assumptions
3. Integer Linear Programming (ILP) model
4. Complexity
5. Conclusion and next steps

## 1. Context and problem statement

### 1.1 Context
ADEME has launched a call for expressions of interest to support pilot projects for new mobility solutions for people and goods. CesiCDP already works on *Smart Multimodal Mobility* with several partners. Cleaner and cheaper transport technologies raise a new challenge: managing resources well. Delivery routing is a central example, with applications in parcel and mail delivery, road-network management and waste collection. Shorter routes mean less time on the road, less exposure to congestion and lower emissions.

### 1.2 Problem statement
A depot, a road network and a set of customers are given. Each customer must be visited once by one of $k$ identical trucks of capacity $Q$. Each truck leaves the depot, serves a subset of customers and returns to the depot. The time needed to drive along a road depends on the **time slot** (expected traffic). We must choose which customers go in which route and the order of the visits so as to **minimize the total duration of the routes**. The method must come from Operations Research and scale to several thousand customers.

### 1.3 Scope
- **Basic version (TD-CVRP):** a fleet of $k$ identical trucks of capacity $Q$, with travel times that depend on the time slot.
- **Advanced version (TD-CVRP-TW):** in addition, each customer has a service time $s_i$ and a delivery time window $[a_i,b_i]$. Delivering outside the window is forbidden, but a truck that arrives early may wait.

We model the advanced version. The basic version is the special case $a_i=0$, $b_i=T$.

## 2. Data and assumptions

### 2.1 Input data
| Data | Symbol | Unit |
|---|---|---|
| Road network (nodes and road segments) | $G=(V,A)$ | directed graph |
| Length of road segment $a$ | $d_a>0$ | km |
| Time slots | $0=\beta_0<\beta_1<\dots<\beta_K=T$, slot $s=[\beta_{s-1},\beta_s)$ | min |
| Speed on segment $a$ during slot $s$ | $v^{s}_a>0$ | km/min |
| Depot and customers | node $0$, $C=\{1,\dots,n\}$ | – |
| Demand, service time and time window of customer $i$ | $q_i$, $s_i$, $[a_i,b_i]$ | units, min, min |
| Number of trucks and capacity | $k$, $Q$ | –, units |
| Planning horizon | $T$ | min |

For the experiments, instances will be generated (random customers, synthetic traffic profiles with morning and evening peaks), and the VRPLIB benchmark instances will be used for validation.

### 2.2 Travel times
- On a road segment, the speed changes when the truck crosses a slot boundary (model of Ichoua et al., 2003). $\tau_a(t)$ is the travel time when entering segment $a$ at time $t$. This model has the **FIFO** property: leaving later never means arriving earlier.
- Between two customers $i$ and $j$, the truck takes the fastest path in $G$. $\tau_{ij}(t)$ is this travel time when leaving $i$ at time $t$.
- **Simplification for the ILP:** $\tau^{s}_{ij}$ is the fastest-path time from $i$ to $j$ computed with the speeds of slot $s$. It is used for the whole trip when the truck leaves $i$ during slot $s$, which keeps the model linear.

### 2.3 Assumptions
| # | Assumption |
|---|---|
| A1 | One depot; at most $k$ identical trucks of capacity $Q$ start and end there. |
| A2 | Each customer is served exactly once by one truck (no split delivery), and $0<q_i\le Q$. |
| A3 | Travel times are deterministic and follow Section 2.2, so FIFO holds. |
| A4 | A truck may drive through a node without serving it, so it always takes fastest paths. |
| A5 | Service must **start** inside $[a_i,b_i]$; a truck that arrives early waits. |
| A6 | All trucks leave the depot at time $0$ and must be back by $T$. |
| A7 | The objective is the total duration of the routes, which by A6 is the sum of the return times. |

## 3. Integer Linear Programming model

To write the routes as an ILP, the depot is split into two nodes: $0$ (start of a route) and $n+1$ (end of a route). Arcs go from $N=C\cup\{0\}$ to $N^{+}=C\cup\{n+1\}$, and $\tau^{s}_{i,n+1}=\tau^{s}_{i0}$.

### 3.1 Decision variables
| Variable | Domain | Meaning |
|---|---|---|
| $x_{ij}$, $i\in N$, $j\in N^{+}$, $i\ne j$ | $\{0,1\}$ | 1 if a truck drives directly from $i$ to $j$ (arc $(0,n+1)$ excluded) |
| $z^{s}_{i}$, $i\in C$, $s\in\{1,\dots,K\}$ | $\{0,1\}$ | 1 if the truck leaves customer $i$ during slot $s$ |
| $r_i$ | $\ge 0$ | arrival time at $i$ |
| $u_i$ | $[a_i,b_i]$ | start of service at $i$ |
| $t_i$ | $[0,T)$ | departure time from $i$ |
| $L_i$ | $[q_i,Q]$ | load delivered on the route up to and including $i$ |
| $f_i$ | $[0,T]$ | return time to the depot if $i$ is the last customer of its route, 0 otherwise |

### 3.2 Objective
$$
\min \sum_{i\in C} f_i \qquad (O)
$$

Only the last customer of each route has $f_i>0$, so (O) is the sum of the return times, which is the total duration of the routes (A6, A7).

### 3.3 Constraints
**Routing**

$$
\sum_{i\in N\setminus\{j\}}x_{ij}=1\quad\forall j\in C \qquad (C1)
$$

$$
\sum_{h\in N^{+}\setminus\{j\}}x_{jh}=1\quad\forall j\in C \qquad (C2)
$$

$$
\sum_{j\in C}x_{0j}\le k \qquad (C3)
$$

Every customer has exactly one predecessor and one successor (C1, C2), and at most $k$ routes leave the depot (C3).

**Capacity and subtour elimination**

$$
L_j\ \ge\ L_i+q_j-Q\,(1-x_{ij})\quad\forall i,j\in C,\ i\ne j \qquad (C4)
$$

If $j$ follows $i$, the load increases by $q_j$; otherwise the constraint is inactive because $L_i\le Q$. This is the capacity form of the Miller–Tucker–Zemlin constraints (Miller et al., 1960; Kulkarni & Bhave, 1985). It also forbids subtours: adding (C4) along a cycle of customers gives $0\ge\sum q_i>0$, which is impossible.

**Time-dependent travel**

$$
\sum_{s=1}^{K}z^{s}_{i}=1,\qquad \sum_{s=1}^{K}\beta_{s-1}z^{s}_{i}\ \le\ t_i\ \le\ \sum_{s=1}^{K}(\beta_s-\varepsilon)\,z^{s}_{i}\quad\forall i\in C \qquad (C5)
$$

$$
t_i\ \ge\ u_i+s_i\quad\forall i\in C \qquad (C6)
$$

$$
r_j\ \ge\ \tau^{1}_{0j}\,x_{0j}\quad\forall j\in C \qquad (C7)
$$

$$
r_j\ \ge\ t_i+\sum_{s=1}^{K}\tau^{s}_{ij}\,z^{s}_{i}-M_{ij}(1-x_{ij})\quad\forall i,j\in C,\ i\ne j \qquad (C8)
$$

(C5) chooses the slot in which the truck leaves $i$; $\varepsilon>0$ is a small time step that makes slots half-open. (C6): the truck leaves after the service. (C7): leaving the depot at time 0 means leaving in slot 1. (C8): if $j$ follows $i$, the arrival at $j$ is at least the departure from $i$ plus the travel time of the departure slot. $M_{ij}=T+\max_s\tau^{s}_{ij}$ is large enough to deactivate the constraint when $x_{ij}=0$. The terms $\tau^{s}_{ij}z^{s}_{i}$ are linear because $\tau^{s}_{ij}$ is a constant.

**Time windows and waiting**

$$
u_i\ \ge\ r_i,\qquad a_i\le u_i\le b_i\quad\forall i\in C \qquad (C9)
$$

The truck may wait until the window opens ($u_i\ge r_i$, $u_i\ge a_i$) but cannot serve after it closes ($u_i\le b_i$). For the basic version, $a_i=0$ and $b_i=T$.

**Return to the depot**

$$
f_i\ \ge\ t_i+\sum_{s=1}^{K}\tau^{s}_{i0}\,z^{s}_{i}-M_i\,(1-x_{i,n+1}),\qquad f_i\le T\,x_{i,n+1}\quad\forall i\in C \qquad (C10)
$$

With $M_i=T+\max_s\tau^{s}_{i0}$: if $i$ is the last customer, $f_i$ is its return time and is at most $T$; otherwise $f_i=0$.

### 3.4 Full model

$$
\begin{aligned}
(O)\quad & \min\ \textstyle\sum_{i\in C} f_i\\
(C1)\quad & \textstyle\sum_{i\in N\setminus\{j\}}x_{ij}=1 \quad \forall j\in C\\
(C2)\quad & \textstyle\sum_{h\in N^{+}\setminus\{j\}}x_{jh}=1 \quad \forall j\in C\\
(C3)\quad & \textstyle\sum_{j\in C}x_{0j}\le k\\
(C4)\quad & L_j\ge L_i+q_j-Q(1-x_{ij}) \quad \forall i,j\in C,\ i\ne j\\
(C5)\quad & \textstyle\sum_{s}z^{s}_{i}=1,\ \ \sum_{s}\beta_{s-1}z^{s}_{i}\le t_i\le\sum_{s}(\beta_s-\varepsilon)z^{s}_{i} \quad \forall i\in C\\
(C6)\quad & t_i\ge u_i+s_i \quad \forall i\in C\\
(C7)\quad & r_j\ge\tau^{1}_{0j}x_{0j} \quad \forall j\in C\\
(C8)\quad & r_j\ge t_i+\textstyle\sum_{s}\tau^{s}_{ij}z^{s}_{i}-M_{ij}(1-x_{ij}) \quad \forall i,j\in C,\ i\ne j\\
(C9)\quad & u_i\ge r_i,\ \ a_i\le u_i\le b_i \quad \forall i\in C\\
(C10)\quad & f_i\ge t_i+\textstyle\sum_{s}\tau^{s}_{i0}z^{s}_{i}-M_i(1-x_{i,n+1}),\ \ f_i\le T\,x_{i,n+1} \quad \forall i\in C\\
& x_{ij}\in\{0,1\},\ z^{s}_{i}\in\{0,1\},\ r_i\ge0,\ t_i\in[0,T),\ q_i\le L_i\le Q,\ f_i\ge0
\end{aligned}
$$

The model has $O(n^2)$ variables and constraints, mostly (C4) and (C8). Because of the big-M constraints its linear relaxation is weak, so it can only be solved exactly on small instances (see Section 4).

## 4. Complexity

### 4.1 Link with classical problems
Each problem in the table adds one feature to the one above, so each one is a special case of the next.

| Problem | Feature added | Reference |
|---|---|---|
| TSP | one vehicle visits every city once and returns | Karp (1972) |
| CVRP | $k$ vehicles of capacity $Q$, customer demands | Dantzig & Ramser (1959); Toth & Vigo (2014) |
| VRPTW | time windows and service times | Solomon (1987) |
| **TD-CVRP-TW** (ours) | travel times depend on the time slot | Ichoua et al. (2003) |

Since our problem contains the TSP as a special case, we expect it to be NP-hard. We prove it by a reduction from the TSP.

### 4.2 Decision version
Complexity classes apply to yes/no questions, so we work with the decision version of our problem:

> **TD-CVRP-TW-D:** given an instance of our problem (Section 2) and a number $B$, is there a feasible solution (A1–A6) whose total duration is at most $B$?

If the optimization problem could be solved in polynomial time, this question could be too (compare the optimum with $B$). So if TD-CVRP-TW-D is NP-hard, the optimization problem is NP-hard as well.

### 4.3 NP-hardness: reduction from the TSP
**Source problem.** We reduce from the TSP with the triangle inequality ($\Delta$-TSP): given $m$ cities, integer costs $c_{ij}=c_{ji}\ge1$ with $c_{ij}\le c_{ih}+c_{hj}$, and a bound $B$, is there a tour visiting every city once of cost at most $B$? This problem is NP-complete (Garey & Johnson, 1979): the classical proof from Hamiltonian Cycle only uses costs 1 and 2, which always satisfy the triangle inequality.

We need the triangle inequality because of A4: trucks always take fastest paths, and fastest-path times always satisfy it. With metric costs, the direct road is a fastest path, so the travel times in our instance are exactly the TSP costs.

**Construction.** From a $\Delta$-TSP instance $(m,c,B)$, we build the following instance of our problem in $O(m^2)$ time:

| Our instance | Value |
|---|---|
| Road network | one node per city, a road in each direction between every pair, $d_{ij}=c_{ij}$ |
| Traffic | one slot ($K=1$) and speed 1 km/min everywhere, so $\tau_{ij}(t)=c_{ij}$ |
| Depot and customers | city $0$ is the depot, the other $n=m-1$ cities are the customers |
| Fleet | $k=1$, $q_i=1$, $Q=n$ |
| Time windows | $s_i=0$, $a_i=0$, $b_i=T=m\cdot\max c_{ij}$ (never binding) |
| Bound | $B$ |

**Correctness.** With one truck, a feasible solution is a single route $(0,\pi_1,\dots,\pi_n,0)$ visiting every customer once, i.e. a tour of the $m$ cities. Its duration is the cost of this tour plus any waiting, and waiting is never needed since $a_i=0$.
- If there is a tour of cost at most $B$, start it at city $0$ and use it as the route without waiting. It respects the capacity ($n=Q$) and the windows and horizon (its cost is at most $m\cdot\max c_{ij}=T$), and its duration is at most $B$.
- If there is a route of duration at most $B$, its order of visits is a tour whose cost is at most the duration, hence at most $B$.

So $\Delta$-TSP $\le_p$ TD-CVRP-TW-D, and **TD-CVRP-TW-D is NP-hard**. The instance we built has no binding time windows and a single slot, so even the **basic version** is NP-hard.

### 4.4 Membership in NP
**Certificate:** the list of routes, each given as the ordered list of its customers (polynomial size).

**Check:** verify that every customer appears exactly once, that there are at most $k$ routes and that every route carries at most $Q$. Then simulate each route from time $0$: arrival $r_j=t_i+\tau_{ij}(t_i)$, start of service $u_j=\max(r_j,a_j)$ with $u_j\le b_j$, departure $t_j=u_j+s_j$. Finally, check that every return time is at most $T$ and that their sum is at most $B$.

Leaving as early as possible is always best, because by FIFO (A3) leaving later never gives an earlier arrival. Each $\tau_{ij}(t)$ can be computed in polynomial time with Dijkstra's algorithm adapted to time-dependent travel times, which is correct under FIFO (Dreyfus, 1969). So the check runs in polynomial time and **TD-CVRP-TW-D is in NP**.

### 4.5 Conclusion
TD-CVRP-TW-D is in NP and NP-hard, so it is **NP-complete**, and the optimization problem is **NP-hard**, already in its basic version. Unless P = NP, no algorithm can solve every instance optimally in polynomial time. The ILP of Section 3 will therefore be used on small instances, to get optimal solutions to compare against, while instances with thousands of customers will need a heuristic or metaheuristic method (next deliverable).

## 5. Conclusion and next steps
- **Done:** a formal statement of the problem, an ILP with capacity, time-dependent travel times and time windows, and a proof that the problem is NP-complete.
- **Next:** choose and implement a metaheuristic, then test it on at least 20 instances and on VRPLIB instances, measuring the gap to the best known solutions and the computation time.

**Questions for the tutor**
- Are the two ILP simplifications acceptable at this stage: all trucks leave at time 0 (A6), and the travel time is fixed by the departure slot (Section 2.2)?
- How many time slots $K$ should we use, and should the traffic profiles be synthetic or based on real data?

## References
- Dantzig, G. B., & Ramser, J. H. (1959). The truck dispatching problem. *Management Science*, 6(1), 80–91.
- Dreyfus, S. E. (1969). An appraisal of some shortest-path algorithms. *Operations Research*, 17(3), 395–412.
- Garey, M. R., & Johnson, D. S. (1979). *Computers and Intractability: A Guide to the Theory of NP-Completeness*. W. H. Freeman.
- Ichoua, S., Gendreau, M., & Potvin, J.-Y. (2003). Vehicle dispatching with time-dependent travel times. *European Journal of Operational Research*, 144(2), 379–396.
- Karp, R. M. (1972). Reducibility among combinatorial problems. In R. E. Miller & J. W. Thatcher (Eds.), *Complexity of Computer Computations* (pp. 85–103). Plenum Press.
- Kulkarni, R. V., & Bhave, P. R. (1985). Integer programming formulations of vehicle routing problems. *European Journal of Operational Research*, 20(1), 58–67.
- Miller, C. E., Tucker, A. W., & Zemlin, R. A. (1960). Integer programming formulation of traveling salesman problems. *Journal of the ACM*, 7(4), 326–329.
- Solomon, M. M. (1987). Algorithms for the vehicle routing and scheduling problems with time window constraints. *Operations Research*, 35(2), 254–265.
- Toth, P., & Vigo, D. (Eds.). (2014). *Vehicle Routing: Problems, Methods, and Applications* (2nd ed.). SIAM.